<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Deep_neural_network_of_L_layers/blob/main/Planar_Data_Classification_with_L_Hidden_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [256]:
import numpy as np
from sklearn.datasets import make_moons
m = 500
X, y = make_moons(
    n_samples=m,
    noise=0.2,
    random_state=42
)

In [257]:
def init_parameter(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) # Removed *0.01
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b

In [258]:
W ,b = init_parameter(dim)

In [259]:
def output(W , b , X , L):
  A = X.T
  activation = [A]
  Zs = []
  def relu(z):
    return np.maximum(0,z)
  def sigmoid(z):
    return 1/(1+np.exp(-z))

  for l in range(1, L-1):
    A_prev = A
    Z = np.dot(W[l],A_prev)+b[l]
    Zs.append(Z)

    A = relu(Z)
    activation.append(A)

  ZL = np.dot(W[L-1],A)+b[L-1]
  Zs.append(ZL)

  AL = sigmoid(ZL)
  return activation , Zs , AL

In [260]:
activation , Zs ,  AL = output(W , b , X , len(dim))
print(len(Zs))

3


In [261]:
def loss( y , AL ):
  m = y.shape[0]
  epsilon = 1e-8 # Small value to prevent log(0)
  temp2 = 0
  for i in range(len(y)):
    # Clip AL values to be within (epsilon, 1 - epsilon) to avoid log(0)
    AL_clipped = np.clip(AL[0, i], epsilon, 1 - epsilon)
    temp = y[i] * np.log(AL_clipped) + (1 - y[i])* np.log(1 - AL_clipped)
    temp2 += temp
  return -temp2/m

In [262]:
loss(y , AL)

np.float64(0.7148214430728757)

In [263]:

def backward(W, b, activations, Zs, AL, Y):

    m = Y.shape[0]
    Y = Y.reshape(1, m)

    L = len(W)

    dW = {}
    db = {}

    dZ = AL - Y

    dW[L] = (1/m) * np.dot(dZ, activations[-1].T)
    db[L] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

    dA = np.dot(W[L].T, dZ)

    for l in reversed(range(1, L)):

        dZ = np.array(dA, copy=True)
        dZ[Zs[l-1] <= 0] = 0

        dW[l] = (1/m) * np.dot(dZ, activations[l-1].T)
        db[l] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

        dA = np.dot(W[l].T, dZ)

    return dW, db

In [264]:
def gradient_descent(W , b , dW , db , lr):

  for l in W.keys():
    W[l] = W[l] - lr * dW[l]
    b[l] = b[l] - lr * db[l]
  return W , b

In [265]:
def model(dim):
  W ,b = init_parameter(dim)
  for i in range(10000):
    L = len(dim)
    activation , Zs , AL = output(W , b , X , L)
    cost = loss(y , AL)
    dW , db = backward(W , b , activation , Zs , AL , y)
    W , b = gradient_descent(W , b , dW , db , 0.5)
    if i % 1000 == 0:
      print(f"Iteration {i}: Cost = {cost}")
  return W , b

In [270]:
dim = [2 , 5 , 4, 1]
W, b = model(dim)

Iteration 0: Cost = 1.2998746970261625
Iteration 1000: Cost = 0.27971324331525355
Iteration 2000: Cost = 0.27970597800116237
Iteration 3000: Cost = 0.27970597487521015
Iteration 4000: Cost = 0.2797059748738339
Iteration 5000: Cost = 0.2797059748738333
Iteration 6000: Cost = 0.27970597487383325
Iteration 7000: Cost = 0.2797059748738332
Iteration 8000: Cost = 0.2797059748738332
Iteration 9000: Cost = 0.2797059748738332


In [271]:
activation , Zs , AL = output(W , b , X , len(dim))
pred = (AL > 0.5)

In [272]:
accuracy = np.mean(pred == y)
print("Model Accuracy: ",accuracy*100,"%")

Model Accuracy:  86.0 %
